# 🏥 CareFlow — Notebook 14: Clinical Quality & Safety Metrics
## Hospital Readmissions, Mortality Rates & Attending Physician Performance Radar

**Author:** CareFlow Clinical Quality Committee  
**Standards:** CMS Hospital Readmissions Reduction Program (HRRP), Leapfrog Hospital Safety Grade  
**Dataset:** `careflow_analytics/output/fct_patient_journeys.parquet`

---

### Objectives:
1. Benchmark 30-day readmission and inpatient mortality rates across acuity tiers.
2. Evaluate attending physician quality metrics (LOS, guideline compliance, speed).
3. Generate multi-axial performance radar charts for medical staff credentialing.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')

fct_df = pd.read_parquet('../careflow_analytics/output/fct_patient_journeys.parquet')
print(f"Loaded {len(fct_df):,} clinical encounters.")
fct_df['primary_physician'].value_counts()


In [ ]:
# 1. Attending Physician Performance Metrics
doc_metrics = fct_df.groupby('primary_physician').agg(
    case_volume=('case_id', 'count'),
    mean_los=('total_los_min', 'mean'),
    median_los=('total_los_min', 'median'),
    loopback_rate=('has_radiology_loopback_rework', 'mean'),
    triage_bypass_rate=('is_triage_bypassed_anomaly', 'mean')
).reset_index()

doc_metrics['loopback_pct'] = (doc_metrics['loopback_rate'] * 100).round(1)
doc_metrics['bypass_pct'] = (doc_metrics['triage_bypass_rate'] * 100).round(1)
doc_metrics['mean_los'] = doc_metrics['mean_los'].round(1)

print("Attending Physician Operational & Quality Performance:")
doc_metrics.sort_values(by='case_volume', ascending=False)


In [ ]:
# 2. Physician Quality Bar Chart
plt.figure(figsize=(10, 5))
sns.barplot(data=doc_metrics[doc_metrics['case_volume'] > 10], x='primary_physician', y='mean_los', palette='mako')
plt.axhline(doc_metrics['mean_los'].mean(), color='#f43f5e', linestyle='--', label=f"Hospital Mean ({doc_metrics['mean_los'].mean():.1f}m)")
plt.title("Attending Physician Average Length of Stay (Minutes)")
plt.xticks(rotation=25)
plt.ylabel("Mean LOS (min)")
plt.legend()
plt.tight_layout()
plt.show()
